In [ ]:
import os, re, json, gc, time, platform, warnings, unicodedata, random
from datetime import datetime
from typing import List, Tuple, Dict, Any

# --- Stabilizačné ENV ---
os.environ.setdefault("PYTORCH_FLASH_ATTENTION", "0")
os.environ.setdefault("PYTORCH_ENABLE_SDPA", "0")
os.environ.setdefault("PYTORCH_CUDA_FUSER_DISABLE_FALLBACK", "1")

import torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline, GenerationConfig
from docx import Document
from striprtf.striprtf import rtf_to_text

# kompatibilná definícia "math-only SDPA" naprieč verziami PyTorch
from contextlib import contextmanager
try:
    from torch.nn.attention import sdpa_kernel as _sdpa_kernel_new, SDPBackend as _SDPBackend
    @contextmanager
    def math_sdpa():
        with _sdpa_kernel_new(_SDPBackend.MATH):
            yield
except Exception:
    try:
        from torch.backends.cuda import sdp_kernel as _sdpa_kernel_old
        @contextmanager
        def math_sdpa():
            with _sdpa_kernel_old(enable_math=True, enable_flash=False, enable_mem_efficient=False):
                yield
    except Exception:
        @contextmanager
        def math_sdpa():
            yield

# =========================
# KONFIGURÁCIA
# =========================
MODE = 'title'

INPUT_DIR   = './Input'
OUTPUT_DIR  = './Output'
THESAURUS_XLSX = './Thesaurus/SK_Local_Thesaurus.xlsx'
THESAURUS_COL  = 'slovak_terms'
STOPWORDS_TXT  = './Input/stopword_dictionary.txt'

# --- Jediný parameter na model: teplota (float) ---
MODEL_TEMPERATURE = {
    "Qwen/Qwen2.5-3B-Instruct": 0.7,
    "google/gemma-2-2b-it":     0.7,
}
# Globálne sampling parametre (spoločné pre všetky modely)
TOP_P = 0.9

# Generácia
MAX_NEW_TOKENS = 96
MAX_TEXT_CHARS = 4000
FORCE_4BIT = False  # necháme vypnuté kvôli stabilite

# Kandidáti
MIN_PROPOSALS = 3
MAX_PROPOSALS = 6
TOPK_REPORT   = 3

# Skórovanie
USE_LLM_RUBRIC = True
ALPHA_HEURISTIC = 0.6
BETA_RUBRIC     = 0.4

# OS-aware pamäťové nastavenia
def _configure_cuda_allocator():
    parts = ["max_split_size_mb:128"]
    if platform.system() == "Linux":
        parts.insert(0, "expandable_segments:True")
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = ",".join(parts)
warnings.filterwarnings("ignore", message=r".*expandable_segments not supported.*")
_configure_cuda_allocator()
random.seed(42)

# =========================
# STOPWORDS
# =========================
def load_stopwords(path=STOPWORDS_TXT):
    try:
        with open(path, 'r', encoding='utf-8') as f:
            return {line.strip().lower() for line in f if line.strip()}
    except FileNotFoundError:
        print(f'[INFO] No stopword file at {path}. Continuing without.')
        return set()
    except Exception as e:
        print(f'[WARN] Failed to load stopwords: {e}')
        return set()
STOPWORDS = load_stopwords()

# =========================
# TEZAURUS (template bias)
# =========================
def strip_accents(s: str) -> str:
    return ''.join(ch for ch in unicodedata.normalize('NFD', s) if not unicodedata.combining(ch))

def _canon_left_of_dash(term: str) -> str:
    return term.split(' - ')[0].strip()

def _split_parenthetical_synonyms(term: str):
    m = re.match(r'^(.*?)(\s*\(([^)]+)\))\s*$', term)
    if not m:
        return term.strip(), []
    base = m.group(1).strip()
    inside = m.group(3).strip()
    syns = [s.strip() for s in re.split(r'[\/,;]', inside) if s.strip()]
    return base, syns

def _wildify(s: str) -> str:
    s = s.strip()
    s = re.sub(r'\s+', r'\\s+', re.escape(s))
    return s.replace(r'\-', r'[-–—]')

def build_term_matcher(term: str):
    canon = _canon_left_of_dash(term)
    base, syns = _split_parenthetical_synonyms(canon)
    pats = [re.compile(r'\b' + _wildify(base) + r'(?:\s*\([^)]*\))?\b', re.IGNORECASE)]
    for syn in syns:
        pats.append(re.compile(r'\b' + _wildify(syn) + r'\b', re.IGNORECASE))
    pats_na = [re.compile(r'\b' + _wildify(strip_accents(base)) + r'(?:\s*\([^)]*\))?\b', re.IGNORECASE)]
    pats_na += [re.compile(r'\b' + _wildify(strip_accents(syn)) + r'\b', re.IGNORECASE) for syn in syns]
    return {'canon': canon, 'patterns': pats, 'patterns_noacc': pats_na, 'synonyms': syns}

def load_thesaurus_terms(xlsx_path=THESAURUS_XLSX, column=THESAURUS_COL) -> List[str]:
    if not os.path.exists(xlsx_path):
        raise FileNotFoundError(f'Thesaurus file not found: {xlsx_path}')
    df = pd.read_excel(xlsx_path, engine='openpyxl')
    if column not in df.columns:
        raise ValueError(f'Column "{column}" not found. Columns: {list(df.columns)}')
    raw = df[column].dropna().astype(str).tolist()
    parts = []
    for cell in raw:
        parts.extend(re.split(r'[,\n;]+', cell))
    seen, terms = set(), []
    for t in parts:
        t = t.strip()
        if not t: continue
        low = t.lower()
        if len(low) == 1 or low in STOPWORDS: continue
        if t not in seen:
            seen.add(t); terms.append(t)
    print(f'[INFO] Thesaurus terms loaded: {len(terms)}')
    return terms

THESAURUS_TERMS = load_thesaurus_terms()
TERM_MATCHERS   = { (m := build_term_matcher(t))['canon'].lower(): m for t in THESAURUS_TERMS }
CANON_TERMS     = list(TERM_MATCHERS.keys())

def terms_matched_in_text(text: str, max_terms: int = 30) -> List[str]:
    hits = {}
    if not text: return []
    t_na = strip_accents(text)
    for canon, m in TERM_MATCHERS.items():
        a = sum(len(p.findall(text)) for p in m['patterns'])
        b = sum(len(p.findall(t_na)) for p in m['patterns_noacc'])
        cnt = max(a, b)
        if cnt > 0:
            hits[canon] = cnt
    ordered = [k for k,_ in sorted(hits.items(), key=lambda kv: (-kv[1], -len(kv[0])))]
    return ordered[:max_terms]

# =========================
# DOKUMENTY
# =========================
_HEADING_NAME_PREFIXES = ('heading', 'nadpis','po-heading-id')
_HEADING_EXACT_NAMES   = {'title','subtitle','toc heading','obsah','po-heading-id_'}
_PAGE_LINE_RE = re.compile(r'^\s*(strana|page)\s+\d+(?:\s*/\s*\d+)?\s*$', re.IGNORECASE)
_RUBRIC_RE    = re.compile(r'([.\-–—]{3,})')

def _gather_header_footer_text(doc: Document):
    lines = []
    for sec in doc.sections:
        for part in (sec.header, sec.footer):
            for p in part.paragraphs:
                t = p.text.strip()
                if t: lines.append(t)
            for tb in part.tables:
                for row in tb.rows:
                    for cell in row.cells:
                        for p in cell.paragraphs:
                            t = p.text.strip()
                            if t: lines.append(t)
    return set(lines)

def _looks_like_heading_style(p):
    try: name = (p.style.name or '').strip().lower()
    except: name = ''
    try: sid  = (getattr(p.style,'style_id','') or '').lower()
    except: sid = ''
    if any(name.startswith(pref) for pref in _HEADING_NAME_PREFIXES): return True
    if name in _HEADING_EXACT_NAMES: return True
    if re.match(r'^heading\d+$', name) or re.match(r'^nadpis\s*\d+$', name): return True
    if re.match(r'^heading\d+$', sid): return True
    return False

def _is_header_footer_like(line: str):
    l = line.strip()
    if not l: return False
    if _PAGE_LINE_RE.match(l): return True
    if _RUBRIC_RE.search(l): return True
    if len(l.split()) <= 6 and l == l.upper() and any(ch.isalpha() for ch in l): return True
    return False

def split_docx_by_headers(path: str) -> List[Tuple[str, str]]:
    try:
        doc = Document(path)
        hf  = _gather_header_footer_text(doc)
        sections, current, buf, full = [], None, [], []
        for p in doc.paragraphs:
            t = p.text.strip()
            if not t: continue
            if t in hf or _is_header_footer_like(t): continue
            if _looks_like_heading_style(p):
                if current is not None and buf:
                    sections.append((current, '\n'.join(buf).strip()))
                current, buf = t, []
            else:
                full.append(t)
                if current is not None:
                    buf.append(t)
        if current is not None and buf:
            sections.append((current, '\n'.join(buf).strip()))
        return [('__DOCUMENT__', '\n'.join(full).strip())] + sections
    except Exception as e:
        print(f'[WARN] DOCX split failed {path}: {e}')
        return []

def _is_heading_line(l: str):
    words = l.split()
    if len(words) <= 8:
        alpha = sum(ch.isalpha() for ch in l)
        if alpha and sum(ch.isupper() for ch in l if ch.isalpha())/alpha >= 0.7: return True
        if re.match(r'^\d+(\.\d+)*\s+\S', l): return True
    return False

def split_rtf_by_headers(path: str) -> List[Tuple[str, str]]:
    try:
        raw = rtf_to_text(open(path,'r',encoding='utf-8').read())
        lines = [ln.strip() for ln in raw.splitlines() if ln.strip()]
        lines = [ln for ln in lines if not _is_header_footer_like(ln)]
        sections, current, buf, full = [], None, [], []
        for ln in lines:
            if _is_heading_line(ln):
                if current is not None and buf:
                    sections.append((current, '\n'.join(buf).strip()))
                current, buf = ln, []
            else:
                full.append(ln)
                if current is not None:
                    buf.append(ln)
        if current is not None and buf:
            sections.append((current, '\n'.join(buf).strip()))
        return [('__DOCUMENT__', '\n'.join(full).strip())] + sections
    except Exception as e:
        print(f'[WARN] RTF split failed {path}: {e}')
        return []

# =========================
# LLM loader – FP16, eager attention, math-only SDPA
# =========================
def load_llm(model_name: str):
    device_map = {"": "cuda:0"} if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32

    qconf = None
    if FORCE_4BIT:
        qconf = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type='nf4',
        )

    kwargs = dict(
        device_map=device_map,
        torch_dtype=dtype,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )
    if qconf is not None:
        kwargs["quantization_config"] = qconf

    model = AutoModelForCausalLM.from_pretrained(model_name, **kwargs)
    tok = AutoTokenizer.from_pretrained(model_name)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    try:
        model.config._attn_implementation = "eager"
    except Exception:
        pass

    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tok,
        device_map=device_map,
        max_new_tokens=MAX_NEW_TOKENS,
        return_full_text=False,
    )
    return pipe, model, tok, f"{model_name}{' (4bit)' if qconf else ''}"

def unload_llm(model, tok, pipe):
    try:
        del model, tok, pipe
    except Exception:
        pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# =========================
# GenerationConfig helper
# =========================
def build_generation_config(pipe, **overrides):
    base = pipe.model.generation_config
    gen = GenerationConfig(**base.to_dict())
    supported = set(gen.to_dict().keys())
    for k, v in overrides.items():
        if k in supported and v is not None:
            setattr(gen, k, v)
    return gen

def llm_call(pipe, prompt: str, *, temperature: float = None, top_p: float = None):
    do_sample = temperature is not None and temperature > 0.0
    gen_cfg = build_generation_config(
        pipe,
        do_sample=do_sample,
        temperature=temperature if do_sample else None,
        top_p=top_p if do_sample else None,
        max_new_tokens=MAX_NEW_TOKENS,
    )
    with math_sdpa():
        out = pipe(prompt, generation_config=gen_cfg)
    return out[0]["generated_text"] if isinstance(out, list) else str(out)

# =========================
# Prompty
# =========================
PROMPTS = [
    ("legal_title",
        "ÚLOHA: Navrhni {nmin}–{nmax} presných PRÁVNYCH NADPISOV (3–12 slov) k textu nižšie.\n"
        "Požiadavky: vecné, právne presné, bez bodky na konci, žiadne úvodzovky.\n"
        "Výstup: vráť IBA čistý JSON zoznam reťazcov, napr. [\"Nadpis A\",\"Nadpis B\"].\n\n"
        "TEZAURUS – PRIO (zhody v texte):\n{pri}\n\n"
        "TEZAURUS – VÝBER:\n{samp}\n\n"
        "TEXT:\n{txt}\n\n"
        "ODPOVEĎ:"),
    ("concise_title",
        "Navrhni {nmin}–{nmax} krátkych a zrozumiteľných nadpisov (3–12 slov) k nasledujúcemu právnemu textu.\n"
        "Bez bodky na konci, žiadne úvodzovky. Vráť IBA JSON list reťazcov.\n\n"
        "Preferuj termíny (zoznam je len pomôcka):\n{pri}\n\n"
        "Text:\n{txt}\n\n"
        "ODPOVEĎ:"),
        ]

RUBRIC_PROMPT = (
    "Vyhodnoť každý navrhnutý nadpis podľa kritérií (0–100): presnosť voči textu, právna primeranosť, jasnosť, rozsah 3–12 slov.\n"
    "Vráť JSON objekt {{\"nadpis\": skóre}} pre všetky položky. Žiadny komentár.\n\n"
    "TEXT:\n{txt}\n\n"
    "NÁVRHY:\n{items}\n\n"
    "ODPOVEĎ:"
)

REFINE_PROMPT = (
    "Uprav nasledujúci nadpis tak, aby presne vystihoval text, mal 3–12 slov, bez bodky a bez úvodzoviek.\n"
    "Ak chýba najdôležitejší pojem z PRIO, vhodne ho doplň.\n\n"
    "PRIO POJMY: {pri}\n"
    "TEXT:\n{txt}\n\n"
    "NADPIS:\n{title}\n\n"
    "VÝSTUP (len finálny nadpis):"
)

# =========================
# Parsing & kandidáti
# =========================
def truncate_text(s: str, max_chars: int = MAX_TEXT_CHARS) -> str:
    s = (s or "").strip()
    if len(s) <= max_chars: return s
    return s[: max_chars//2] + "\n...\n" + s[- max_chars//2 :]

def parse_titles(raw: str) -> List[str]:
    s = raw.strip()
    try:
        start = s.index('['); end = s.rindex(']'); blob = s[start:end+1]
        js = json.loads(blob)
        out = [str(x).strip() for x in js if isinstance(x, (str,))]
    except Exception:
        lines = [re.sub(r'^[\-\d\.\)\s]+','', ln).strip() for ln in s.splitlines() if ln.strip()]
        out = lines
    cleaned = []
    for o in out:
        o = re.sub(r'[\.，。…]+$','', o).strip(' "„”')
        if 3 <= len(o.split()) <= 12:
            cleaned.append(o)
    return cleaned

def collect_candidates(pipe, model_name: str, text: str, prio: List[str], samp: List[str]) -> List[str]:
    temp = MODEL_TEMPERATURE.get(model_name, 0.7)  # jediný parameter
    txt = truncate_text(text)
    pri = "\n".join(prio[:20]) if prio else "(žiadne priame zhody)"
    samp = "\n".join(samp[:40])
    bag = []
    for key, tpl in PROMPTS:
        prompt = tpl.format(nmin=MIN_PROPOSALS, nmax=MAX_PROPOSALS, pri=pri, samp=samp, txt=txt)
        raw = llm_call(pipe, prompt, temperature=temp, top_p=TOP_P)
        bag.extend(parse_titles(raw))
    # dedupe (accent-insensitive)
    seen, uniq = set(), []
    for t in bag:
        k = strip_accents(t.lower())
        if k not in seen:
            seen.add(k); uniq.append(t)
    return uniq

# =========================
# Heuristický skór + rubric + refine
# =========================
def _tokenize_words(s: str) -> List[str]:
    return re.findall(r"[A-Za-zÁÄČĎÉÍĽŇÓÔŔŠŤÚÝŽáäčďéíľňóôŕšťúýž0-9]+", s.lower())

def _ngrams(tokens: List[str], n: int) -> List[str]:
    return [" ".join(tokens[i:i+n]) for i in range(len(tokens)-n+1)] if len(tokens)>=n else []

def simple_score(title: str, src_text: str, priority_terms: List[str]) -> float:
    if not title: return 0.0
    words = title.split()
    n = len(words)
    len_score = 1.0 if 4 <= n <= 10 else max(0.0, 1.0 - abs(n - 7)*0.12)
    t_na = strip_accents(title.lower())
    cov_hits, cov_weight = 0.0, 0.0
    for term in priority_terms[:20]:
        term_na = strip_accents(term.lower()); w = min(1.5, 0.6 + 0.1*len(term_na.split()))
        cov_weight += w
        if term_na in t_na: cov_hits += w
    cov_score = (cov_hits / cov_weight) if cov_weight>0 else 0.0
    src_tok = _tokenize_words(src_text); tit_tok = _tokenize_words(title)
    src_un, src_bi = set(_ngrams(src_tok,1)), set(_ngrams(src_tok,2))
    tit_un, tit_bi = set(_ngrams(tit_tok,1)), set(_ngrams(tit_tok,2))
    un_overlap = len(tit_un & src_un)/max(1,len(tit_un))
    bi_overlap = len(tit_bi & src_bi)/max(1,len(tit_bi))
    ngram_score = 0.35*un_overlap + 0.65*bi_overlap
    non_stop = [w for w in tit_tok if w not in STOPWORDS]
    dens = len(non_stop)/max(1,len(tit_tok))
    dens_score = 0.3 + 0.7*dens
    p = 0.0
    if title.strip().endswith(('.', '…')): p += 0.12
    if sum(ch.isupper() for ch in title) > 0.7*sum(ch.isalpha() for ch in title if ch.isalpha()): p += 0.12
    if re.match(r"^(úvod|všeobecné ustanovenia)$", title.strip().lower()): p += 0.1
    score = 0.4*len_score + 0.35*cov_score + 0.2*ngram_score + 0.15*dens_score - p
    return max(0.0, min(1.0, score))

RUBRIC_PROMPT = (
    "Vyhodnoť každý navrhnutý nadpis podľa kritérií (0–100): presnosť voči textu, právna primeranosť, jasnosť, rozsah 3–12 slov.\n"
    "Vráť JSON objekt {{\"nadpis\": skóre}} pre všetky položky. Žiadny komentár.\n\n"
    "TEXT:\n{txt}\n\n"
    "NÁVRHY:\n{items}\n\n"
    "ODPOVEĎ:"
)

def rubric_score(pipe, model_name: str, titles: List[str], src_text: str) -> Dict[str, float]:
    if not USE_LLM_RUBRIC or not titles:
        return {t: 0.5 for t in titles}
    items = "\n".join(f"- {t}" for t in titles)
    prompt = RUBRIC_PROMPT.format(txt=truncate_text(src_text), items=items)
    temp = MODEL_TEMPERATURE.get(model_name, 0.7)
    raw = llm_call(pipe, prompt, temperature=temp, top_p=TOP_P)
    try:
        start = raw.index('{'); end = raw.rindex('}')
        js = json.loads(raw[start:end+1])
        scores = {}
        for k,v in js.items():
            try:
                val = float(v); scores[k.strip()] = max(0.0, min(1.0, val/100.0))
            except Exception:
                continue
        for t in titles:
            scores.setdefault(t, 0.5)
        return scores
    except Exception:
        return {t: 0.5 for t in titles}

REFINE_PROMPT = (
    "Uprav nasledujúci nadpis tak, aby presne vystihoval text, mal 3–12 slov, bez bodky a bez úvodzoviek.\n"
    "Ak chýba najdôležitejší pojem z PRIO, vhodne ho doplň.\n\n"
    "PRIO POJMY: {pri}\n"
    "TEXT:\n{txt}\n\n"
    "NADPIS:\n{title}\n\n"
    "VÝSTUP (len finálny nadpis):"
)
def refine_title(pipe, model_name: str, title: str, text: str, prio: List[str]) -> str:
    prompt = REFINE_PROMPT.format(pri=", ".join(prio[:10]) if prio else "(žiadne)",
                                    txt=truncate_text(text), title=title)
    temp = MODEL_TEMPERATURE.get(model_name, 0.7)
    raw = llm_call(pipe, prompt, temperature=temp, top_p=TOP_P)
    out = raw.strip().splitlines()[0].strip(' "„”')
    out = re.sub(r'[\.，。…]+$','', out).strip()
    if len(out.split()) < 3 or len(out.split()) > 12:
        return title
    return out

# =========================
# Hlavná logika pre jeden model
# =========================
def run_on_model(model_name: str, sections: List[Tuple[str,str,str]]) -> List[Dict[str,Any]]:
    print(f"\n[MODEL] Loading {model_name} ...")
    t0 = time.perf_counter()
    pipe, model, tok, label = load_llm(model_name)
    print(f"[MODEL] Loaded {label} in {time.perf_counter()-t0:.2f}s")

    rows = []
    for (f, header, text) in sections:
        prio = terms_matched_in_text(text, max_terms=30)
        samp = prio if prio else CANON_TERMS[:40]
        cands = collect_candidates(pipe, model_name, text, prio, samp)
        if not cands:
            rows.append({"model": label, "file": f, "header": header,
                            "final_title": "", "final_score": 0.0,
                            "top3": "", "n_candidates": 0})
            continue

        hscores = {t: simple_score(t, text, prio) for t in cands}
        rscores = rubric_score(pipe, model_name, cands[:24], text)

        combined = []
        for t in cands:
            h = hscores.get(t, 0.0); r = rscores.get(t, 0.5)
            s = ALPHA_HEURISTIC*h + BETA_RUBRIC*r
            combined.append((t, s, h, r))
        combined.sort(key=lambda x: x[1], reverse=True)

        top3 = combined[:TOPK_REPORT]
        winner = top3[0][0]
        refined = refine_title(pipe, model_name, winner, text, prio)
        final_score = simple_score(refined, text, prio)

        rows.append({
            "model": label,
            "file": f,
            "header": header,
            "final_title": refined,
            "final_score": round(final_score, 3),
            "top3": " | ".join(f"{t} (c={round(s,3)}/h={round(h,3)}/r={round(r,3)})" for t,s,h,r in top3),
            "n_candidates": len(cands)
        })

    unload_llm(model, tok, pipe)
    return rows

# =========================
# Driver
# =========================
def collect_sections(input_dir=INPUT_DIR) -> List[Tuple[str,str,str]]:
    out = []
    files = sorted(os.listdir(input_dir), key=str.lower)
    for f in files:
        p = os.path.join(input_dir, f)
        if f.lower().endswith('.docx'):
            secs = split_docx_by_headers(p)
        elif f.lower().endswith('.rtf'):
            secs = split_rtf_by_headers(p)
        else:
            continue
        for header, text in secs:
            if text and text.strip():
                out.append((f, header, text))
    return out

def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    sections = collect_sections(INPUT_DIR)
    if not sections:
        print("[ERROR] No sections found.")
        return

    all_rows = []
    for m in MODEL_TEMPERATURE.keys():
        all_rows.extend(run_on_model(m, sections))

    today = datetime.today().strftime("%Y-%m-%d")
    df = pd.DataFrame(all_rows, columns=[
        "model","file","header","final_title","final_score","top3","n_candidates"
    ])
    xlsx_path = os.path.join(OUTPUT_DIR, f"titles_qwen_gemma_{today}.xlsx")
    csv_path  = os.path.join(OUTPUT_DIR, f"titles_qwen_gemma_{today}.csv")
    try: df.to_excel(xlsx_path, index=False, engine="openpyxl"); print(f"[SAVED] {xlsx_path}")
    except Exception as e: print(f"[WARN] Excel write failed: {e}")
    try: df.to_csv(csv_path, index=False, encoding="utf-8-sig"); print(f"[SAVED] {csv_path}")
    except Exception as e: print(f"[WARN] CSV write failed: {e}")

    print("\n===== SUMMARY (avg by model) =====")
    print(df.groupby("model")["final_score"].mean().round(3))

if __name__ == "__main__":
    main()


[INFO] Thesaurus terms loaded: 3000

[MODEL] Loading Qwen/Qwen2.5-3B-Instruct ...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0


[MODEL] Loaded Qwen/Qwen2.5-3B-Instruct in 4.06s
